# Evergreen Python 백테스트 노트북

이 노트북은 Python 백테스트의 README이자 스모크 테스트다. 기본 실행은 v1부터 v5까지 모든 전략을 공식 Upbit SDK 일봉 데이터로 비교한다.

실제 Upbit 일봉 데이터는 `outputs/data/upbit-cache`에 CSV로 캐시된다. 최초 실행은 공개 캔들 API를 호출하고, 같은 기간/마켓은 다음부터 캐시를 먼저 읽는다.

현재 공식 SDK는 캔들/주문/계좌 같은 OpenAPI 리소스를 제공하고, RSI/EMA/ATR 같은 전략 지표 계산기는 제공하지 않는다. 그래서 데이터 조회는 SDK를 쓰고, 지표 계산은 버전별 전략 코드와 Java 전략 엔진에서 같은 계약으로 유지한다.


## 전략 설명서

### 공통 연동정의

모든 전략은 같은 입력과 출력 계약을 따른다.

- 입력: `CandleBar[]`, `signal_index`, 현재 포지션(`qty`, `avgPrice`, `updatedAt`, `positionRatio`), 버전별 파라미터 JSON
- 출력: `StrategyEvaluation`
- 실행 판단: `decision.action`은 `BUY`, `SELL`, `HOLD` 중 하나
- 주문 비중: `decision.targetPositionRatio`는 `evergreen.trading.signal-order-notional` 기준 목표 포지션 비중이다. 0.0은 전량 현금, 1.0은 기준 금액만큼 보유다. v3처럼 변동성 비중 전략은 `max_leverage`까지 값을 낼 수 있으며, LIVE 현물 주문은 실제 가용 KRW 안에서만 증액한다.
- 판단 이유: `decision.signalReason`은 매수/매도/유지의 원인을 기록한다.
- 진단값: `diagnostics`는 전략별 보조 지표다. 실행 판단의 필수 계약이 아니라 로그, 차트, 디버깅용이다.

Java에서는 v1~v5가 모두 `TradingStrategyEngine`으로 등록되고, Python 계약 파일도 같은 `StrategyInput -> StrategyEvaluation` 구조를 내보낸다. 전략별 설정은 JSON/설정값으로 넣고, 출력은 항상 `action`, `signalReason`, `targetPositionRatio`, `diagnostics`로 읽으면 된다.

Java 주문 계층은 `targetPositionRatio`를 목표 포지션 비중으로 받아 처리한다. 현재 비중은 보유 수량과 신호 캔들 종가를 `signal-order-notional` 기준 금액과 비교해 계산한다. 0.0은 전량 매도, 1.0은 기준 금액만큼 매수이며, 중간 비중은 보유 수량을 비례 매도하거나 증액 매수 금액을 조정한다. LIVE 모드의 증액 매수는 실제 가용 KRW를 넘지 않는다.

### v1: MA + RSI

가장 단순한 추세 추종 베이스라인이다. 가격이 이동평균 위에 있고 이동평균 기울기가 상승이며 RSI가 설정값보다 낮을 때 매수하고, 가격이 이동평균 아래로 내려가면 매도한다. 포지션은 0 또는 1의 전량 진입/전량 청산이다. 주요 조정값은 `rsi_buy`, `ma_len`, `ma_slope_days`다.

### v2: Regime + ATR Stop

EMA 밴드로 상승/하락 국면을 나누고 ATR 추적 손절을 붙인 전략이다. `BEAR`에서 `BULL`로 전환될 때 매수하고, `BULL`에서 `BEAR`로 전환되거나 ATR 추적 손절이 발생하면 매도한다. 주요 조정값은 `regime_ema_len`, `atr_period`, `atr_trail_multiplier`, `regime_band`다.

### v3: Volatility Target + Regime

v2 구조에 변동성 기반 목표 비중 조절을 추가한 전략이다. 상승 국면에서 목표 변동성에 맞춰 현재 비중보다 목표 비중이 커지면 증액하고, 하락 국면이나 ATR 추적 손절 또는 목표 비중 축소가 필요하면 감액한다. 포지션은 0.0부터 `max_leverage`까지의 동적 목표 비중이다. 주요 조정값은 `vol_target`, `max_leverage`, `min_exposure`와 레짐/ATR 파라미터다.

### v4: Weekly Filter

v2에 주간 EMA 필터를 추가한 전략이다. 일봉 레짐이 매수 신호를 내고 주간 필터도 상승으로 판단할 때만 진입한다. 매도는 일봉 레짐 매도 또는 ATR 추적 손절 기준을 따른다. 주요 조정값은 `weekly_ema_len`과 v2 계열 파라미터다.

### v5: Adaptive ATR Exit

v2의 ATR 추적 손절을 변동성 국면에 따라 다르게 적용하는 전략이다. `BEAR`에서 `BULL`로 전환될 때 매수하고, `BULL`에서 `BEAR`로 전환되거나 현재 변동성 국면에 맞춘 ATR 추적 손절이 발생하면 매도한다. 주요 조정값은 `atr_mult_low_vol`, `atr_mult_high_vol`, `vol_regime_lookback`, `vol_regime_threshold`, `regime_band`다.

### 선택 기준

- 단순 기준선이 필요하면 v1
- 안정적인 레짐/손절 베이스라인이 필요하면 v2
- 비중 조절까지 실험하려면 v3
- 큰 추세 필터를 강하게 걸고 싶으면 v4
- 변동성 국면별 손절을 보고 싶으면 v5


## 1. 프로젝트 경로와 커널 확인

IntelliJ나 Jupyter를 어디서 열어도 import가 되도록 프로젝트 루트를 찾고 `PYTHONPATH`에 추가한다. IntelliJ에서는 프로젝트 인터프리터를 `.venv`로 선택하고 커널은 `Python 3`를 쓰면 된다. 다른 위치에서 실행해야 하면 환경 변수 `EVERGREEN_PROJECT_ROOT`에 프로젝트 루트를 넣으면 된다.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def 프로젝트_루트_찾기() -> Path:
    env_root = os.environ.get("EVERGREEN_PROJECT_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    candidates = [Path.cwd(), *Path.cwd().parents, Path("/Users/moon/IdeaProjects/evergreen")]
    for candidate in candidates:
        if (candidate / "settings.gradle").exists() and (candidate / "evergreen_backtest/strategies/v5.py").exists():
            return candidate.resolve()
    raise RuntimeError("Evergreen 프로젝트 루트를 찾지 못했다. EVERGREEN_PROJECT_ROOT를 설정해라.")


PROJECT_ROOT = 프로젝트_루트_찾기()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs" / ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "outputs" / ".cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

for candidate in (PROJECT_ROOT,):
    text = str(candidate)
    if text not in sys.path:
        sys.path.insert(0, text)

print("프로젝트 루트:", PROJECT_ROOT)
print("Python:", sys.executable)

## 2. 실행 API 불러오기

전략 버전은 `evergreen_backtest/strategies/v*.py`에서 자동으로 찾고, 실행은 `evergreen_backtest` 패키지가 담당한다.

In [ ]:
from datetime import datetime, timezone
from pprint import pprint

from evergreen_backtest import BacktestRunRequest, available_versions, run_backtest
from evergreen_backtest.plotting import plot_equity_curves

print("사용 가능한 전략 버전:", ", ".join(available_versions()))

## 3. 여기만 바꿔서 실행

- `실행할_버전`: 기본값 `("all",)`은 v1~v5 전체 실행이다. 특정 버전만 보려면 `("v5",)`처럼 넣는다.
- `공통_설정`: 모든 버전에 적용한다. 각 버전이 모르는 항목은 자동으로 무시된다.
- `버전별_설정`: 특정 버전에만 적용한다. 오타나 지원하지 않는 항목은 바로 에러를 낸다.


In [ ]:
실행할_버전 = ("all",)
마켓 = "KRW-BTC"
시작일 = datetime(2020, 1, 1, tzinfo=timezone.utc)
종료일 = None

공통_설정 = {
    "top_k": 5,
    "grid_parallelism": 1,
}

버전별_설정 = {
    "v5": {
        # 예: "atr_period": 14,
    },
}


## 4. 백테스트 실행

이 셀이 노트북의 테스트 역할을 한다. 기본값에서는 v1~v5 전체 결과가 모두 나와야 한다.


In [ ]:
요청 = BacktestRunRequest(
    versions=실행할_버전,
    market=마켓,
    from_dt=시작일,
    to_dt=종료일,
    cache_dir=PROJECT_ROOT / "outputs" / "data" / "upbit-cache",
    common_config=공통_설정,
    version_config=버전별_설정,
    output_dir=PROJECT_ROOT / "outputs" / "backtests" / "latest",
)

결과 = run_backtest(요청)
요약 = 결과.summary_dataframe()
예상_버전 = set(available_versions()) if 실행할_버전 == ("all",) else set(실행할_버전)

assert not 요약.empty, "백테스트 요약이 비어 있다."
assert 예상_버전.issubset(set(요약["version"])), "요청한 버전 중 누락된 결과가 있다."

요약


## 5. 자산 곡선 보기

`phase`는 `validation`, `test`, `full` 중 하나를 넣을 수 있다. 기본 비교 구간은 최적화 이후 검증용으로 쓰는 `test`다.

In [ ]:
그림 = plot_equity_curves(
    결과,
    phase="test",
    save_to=PROJECT_ROOT / "outputs" / "backtests" / "latest" / "equity_test.png",
)
그림

## 6. Java에서 바로 쓰는 계약 파일

`strategy_contracts.json`에는 계약 스키마 버전, 공통 StrategyInput/StrategyEvaluation 정의, 버전별 선택 파라미터, 마지막 평가의 `decision.action`(`BUY`/`SELL`/`HOLD`), `decision.signalReason`, `decision.targetPositionRatio`, 검증/테스트/전체 성과 요약이 들어간다. `javaInteropReady`는 v1~v5가 같은 Java/Python 연동정의를 만족한다는 뜻이다. Java 쪽에서는 `javaParamsCamelCase`를 버전별 설정 객체에 매핑하면 된다. `selectedParamsCamelCase`는 Python 백테스트 비용 파라미터까지 포함한 전체 선택값이다. Java 주문 실행은 `signal-order-notional` 기준 `targetPositionRatio`로 전량 진입/청산과 중간 비중 조정을 같은 계약으로 처리한다.


In [ ]:
출력_경로 = 결과.write_outputs()
계약 = 결과.contracts()

print(f"출력 경로: {출력_경로}")
print("\nJava 전달용 파라미터와 마지막 평가 예시")
for 버전, payload in 계약["versions"].items():
    print("-", 버전, "Java 연동:", payload["javaInteropReady"])
    pprint(payload["javaParamsCamelCase"])
    decision = payload["lastEvaluation"]["decision"]
    print("  마지막 평가:", decision["action"], decision["signalReason"], "target=", decision["targetPositionRatio"])


## 7. 실제 Upbit 데이터로 실행할 때

공식 SDK를 쓰려면 의존성을 동기화한 뒤 `.venv`의 `Python 3` 커널로 열고 위에서부터 순서대로 실행한다.

```bash
uv sync
uv run jupyter lab backtest_playground.ipynb
uv run jupyter notebook backtest_playground.ipynb
```

최초 실행은 Upbit 일봉 API를 호출하고, 같은 기간/마켓은 다음부터 CSV 캐시를 먼저 읽는다. API 키가 필요한 주문/계좌 기능은 여기서 사용하지 않고, 공개 캔들 데이터만 조회한다.
